# 04: Topology

*Does the graph's structure reflect meaningful biological organisation?*

| Sub-metric | Definition | Quantification | Scoring |
|---|---|---|---|
| **Graph cohesion** | Share of nodes in the largest connected component | LCC nodes / total nodes | direct, 0-1 |
| **Clustering coefficient** | Mean local clustering relative to a density-matched (Erdős-Rényi) random graph | C from a fixed 10,000-node sample per graph; C_ER = \<k\>/(n-1) | log-scaled: min(1, log(C/C_ER) / log(1000)) |
| **Graph reachability** | Whether validated drug-disease pairs stay connected under edge removal, relative to type-matched random pairs | Resilience Index for gold and random pairs over 0-50% dropout | (RI_gold - RI_rand) / max(eps, 1 - RI_rand), clipped at 0 |
| **Community purity** | Alignment of Louvain communities with entity types | Louvain at resolution 1.0 | AMI against canonical entity-type labels, 0-1 |

The dimension score is the equal-weighted mean of the four.

**Reported, not scored.** NMI and ARI beside AMI, and the composition of the largest communities as a curator aid.

Reachability uses the same gold standard as notebook 01 (`data/reference_sets/relation_drug_disease.tsv`, Open Targets approved indications), resolved through the shared `IdResolver`.

**Inputs:** `config/kgs.yaml` · `config/params.yaml` · `data/reference_sets/` · `data/crosswalks/` · KG edge and node files

**Outputs:** `results/{figures,tables}/04_topology/` · `results/checkpoints/04_topology.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

Every graph in `config/kgs.yaml` is loaded as a simple graph via `build_graph`, because cohesion, clustering, modularity and Adamic-Adar are all defined that way. Clustering is estimated from a fixed `cluster_sample_n` sample per graph; every other metric is exact.

In [1]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import pickle
import sys
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

import numpy as np
import pandas as pd
import networkx as nx

from biokgsuite.graph.graph_utils import build_graph
from biokgsuite.io import Outputs, assert_full_panel, find_config, load_config, load_kg, require
from biokgsuite.io.resolve import IdResolver
from biokgsuite.stats import topology as topo
from biokgsuite.stats.coverage import load_reference_sets
from biokgsuite.viz.dimension04 import (fig04_clustering, fig04_cohesion,
                                        fig04_recovery, fig04_resilience)
from biokgsuite.viz.style import setup_style

setup_style()

In [2]:
# Parameters are read with require(), never params.get(key, default): a silent
# default lets a run use a value config/params.yaml never mentions.
config    = load_config(find_config(_root))
params    = config['analysis_params']
BASE      = config['_base_dir']
SEED      = require(params, 'random_seed')
CLUSTER_N = require(params, 'cluster_sample_n')

# Every figure, table and checkpoint goes through `out`, so results land in
# results/{figures,tables}/04_topology/ and
# results/checkpoints/04_topology.pkl.
out = Outputs('04_topology', BASE)

# A run that scores fewer graphs than config/kgs.yaml declares is a
# different experiment, not a smaller one, so it has to be asked for.
ALLOW_PARTIAL_PANEL = False


### Loading

In [3]:
# Free any KG state left by an earlier notebook before loading again, so the
# two sets never coexist in the kernel.
for _v in ('kgs', 'graphs'):
    globals().pop(_v, None)
gc.collect()

kgs, graphs = {}, {}
for name in config['knowledge_graphs']:
    try:
        kg, nodes = load_kg(name, config)
        n_rel = kg['relation'].nunique()
        graphs[name] = build_graph(kg, nodes)
        kgs[name] = {'nodes': nodes}     # edge table freed; only the graph is needed
        del kg
        gc.collect()
        print(f'{name:12s}: {graphs[name].number_of_nodes():>8,} nodes  '
              f'{graphs[name].number_of_edges():>10,} edges  {n_rel:>4} relation types')
    except FileNotFoundError:
        print(f'{name:12s}: [NOT FOUND]')
    except Exception as e:
        print(f'{name:12s}: [ERROR] {e}')

KG_NAMES = list(graphs)
ENTITY_TYPE_MAP = {n: config['knowledge_graphs'][n].get('entity_types', {}) for n in KG_NAMES}
# Refuse to score a panel that is missing graphs.
#
# The broad `except` above keeps one unreadable graph from taking down the run,
# but the cost is that a run finding two of six completes, prints a dimension
# score and writes a checkpoint indistinguishable from a full one. Scores that
# intersect or normalise across the panel then differ from what a full run would
# give, silently. Set ALLOW_PARTIAL_PANEL in the config cell to score a subset
# deliberately; the reduced panel is recorded in the checkpoint either way.
PANEL = assert_full_panel(KG_NAMES, config,
                          allow_partial=ALLOW_PARTIAL_PANEL,
                          notebook='04_topology')


primekg     :  129,375 nodes   4,049,642 edges    30 relation types


hetionet    :   47,031 nodes   2,107,709 edges    24 relation types


drkg        :   97,238 nodes   4,400,766 edges   107 relation types


openbilink  :  185,929 nodes   3,200,327 edges    28 relation types


biokg       :  105,524 nodes   2,043,846 edges    17 relation types
  matrix: loading from cache _cache_matrix_7365ba96_kg.parquet / _cache_matrix_7365ba96_nodes.parquet


  matrix: cached load — 7,354,612 nodes, 81,149,812 edges


matrix      : 7,354,612 nodes  54,038,933 edges    91 relation types
04_topology: panel complete, 6/6 graphs loaded


## 1. Graph cohesion

LCC size is the canonical structural integrity metric for complex networks. A well-curated KG should place >95% of nodes in a single largest connected component (LCC), no single canonical threshold exists, but full LCC connectivity is used as an implicit curation quality signal in practice. Isolated nodes (degree 0) represent entities added to the node list but never linked by any edge.

In [4]:
conn_df, LCC_SETS = topo.connectivity(graphs)
out.table(conn_df.reset_index(), 'graph_cohesion')
display(conn_df)

,Nodes,LCC nodes,LCC fraction,LCC (%),Components,Isolated,Top types outside LCC
KG,,,,,,,
primekg,129375,129375,1.000000,100.00,1,0,-
hetionet,47031,45158,0.960175,96.02,1874,1873,"Gene (1,800), Side Effect (33), Symptom (23)"
drkg,97238,96420,0.991588,99.16,303,0,"Compound (421), Gene (378), Disease (11)"
openbilink,185929,185879,0.999731,99.97,2,0,Anatomy (50)
biokg,105524,98456,0.933020,93.30,1451,0,"Gene/Protein (5,738), Pathway (1,157), Drug (87)"
matrix,7354612,7316507,0.994819,99.48,17524,0,"disease (20,206), clinicalattribute (3,916), p..."


In [5]:
out.figure(fig04_cohesion(conn_df), 'graph_cohesion');

  saved graph_cohesion.pdf and .png (600 dpi)


## 2. Clustering coefficient

The **mean local clustering coefficient C** quantifies the tendency of nodes to form densely connected neighbourhoods, the degree to which a node's neighbours are also connected to each other.

We compare the empirical clustering coefficient against that of an Erdős-Rényi random graph of the same density (C_ER = ⟨k⟩/(n−1)) to obtain a normalised ratio C/C_ER. Values ≫ 1 indicate clustering far above the density-matched expectation; small-worldness additionally requires short path lengths, which are not measured here with dense local neighbourhoods; values near 1 suggest random wiring.

In [6]:
clust_df = topo.clustering(graphs, seed=SEED, k=CLUSTER_N)
out.table(clust_df.reset_index(), 'clustering_coefficient')
display(clust_df)
print(f'Clustering estimated from a fixed {CLUSTER_N:,}-node sample per '
      f'graph (seed {SEED}), applied uniformly regardless of graph size. '
      f'`SW score SE` is the resulting uncertainty on the sub-metric.')

,Nodes sampled,Sampled (%),C (empirical),C SE,C_ER (random),C / C_ER,SW score,SW score SE
KG,,,,,,,,
primekg,10000,7.7,0.0742,0.0017,0.000484,153.3,0.7285,0.0033
hetionet,10000,21.3,0.1092,0.0018,0.001906,57.3,0.5860,0.0023
drkg,10000,10.3,0.2336,0.0030,0.000931,251.0,0.7999,0.0019
openbilink,10000,5.4,0.1113,0.0025,0.000185,601.0,0.9263,0.0032
biokg,10000,9.5,0.1396,0.0025,0.000367,380.3,0.8601,0.0026
matrix,10000,0.1,0.1637,0.0032,0.000002,81948.3,1.0000,0.0000


Clustering estimated from a fixed 10,000-node sample per graph (seed 42), applied uniformly regardless of graph size. `SW score SE` is the resulting uncertainty on the sub-metric.


In [7]:
out.figure(fig04_clustering(clust_df), 'clustering_coefficient');

  saved clustering_coefficient.pdf and .png (600 dpi)


## 3. Graph reachability (external gold standard)

Using **Open Targets approved drug indications** as an external gold standard, the same set
notebook 01 scores against, we measure whether the KG's topology structurally encodes
clinically validated therapeutic relationships. The size of the set is printed below rather
than quoted here, so it cannot drift from the file the run actually reads.

Two complementary analyses:

1. **Recovery curves**, what fraction of gold-standard pairs are reachable at 1-5 hops,
   compared with baseline KG size and density.

2. **Differential Resilience**, how well gold-standard pairs are *preserved* under
   progressive random edge dropout (0-50 %), relative to random pairs. This density-invariant
   score rewards structural specificity, not bulk connectivity.

In [8]:
# Reachability gold standard: the reference set notebook 01 scores against.
#
# This notebook used to build its own drug-disease standard from
# data/gold_standards/, a directory that no longer exists, with hand-written
# MONDO/DOID/MeSH bridges per graph. Two independent drug-disease standards in
# one benchmark cannot be defended, and the per-graph bridges made the result
# depend on how much mapping work had been done for each graph. IdResolver
# dispatches on the identifier's own namespace, so every graph gets the same
# crosswalks. The rule behind the set is in data/REFERENCE_SETS.md.
REFS = load_reference_sets(BASE / 'data' / 'reference_sets')
GOLD_PAIRS = sorted(REFS.relations['drug_disease'])      # (ChEMBL drug, MONDO disease)

_XW = BASE / 'data' / 'crosswalks'
if not _XW.exists():
    raise FileNotFoundError(f'{_XW} missing. Run:  python tools/build_crosswalks.py')
resolver = IdResolver(_XW, verbose=False)

MAX_HOPS = 5
print(f'Gold standard: {len(GOLD_PAIRS):,} approved drug-disease pairs, '
      f'the same set notebook 01 scores.')

Gold standard: 7,254 approved drug-disease pairs, the same set notebook 01 scores.


In [9]:
# Map the gold pairs onto node indices, restricted to the largest component.
# A pair with an endpoint outside the LCC is unreachable by construction, which
# would otherwise be recorded as a topology failure rather than as the absence
# it is.
INDEX_MAPS, mapped_pairs, rows = {}, {}, []
for name in KG_NAMES:
    INDEX_MAPS[name] = topo.canonical_index_maps(
        kgs[name]['nodes'], ENTITY_TYPE_MAP[name], resolver)
    mapped_pairs[name] = topo.map_gold_pairs(
        GOLD_PAIRS, INDEX_MAPS[name], LCC_SETS[name])
    rows.append({
        'KG': name, 'Gold pairs': len(GOLD_PAIRS),
        'Drugs resolved': len(INDEX_MAPS[name]['Drug']),
        'Diseases resolved': len(INDEX_MAPS[name]['Disease']),
        'Pairs mapped (in LCC)': len(mapped_pairs[name]),
        'Mapped (%)': round(100 * len(mapped_pairs[name]) / len(GOLD_PAIRS), 1),
    })

mapping_df = pd.DataFrame(rows).set_index('KG')
out.table(mapping_df.reset_index(), 'gold_pair_mapping')
display(mapping_df)
# The resolver's crosswalks are several hundred MB and are needed only here.
# Released before the dropout section, which holds every graph plus large
# transient arrays; the two together exceed memory on a small machine.
del resolver, INDEX_MAPS
gc.collect()

print('Unmapped pairs are a coverage fact, not a topology one: the graph does not')
print('contain that drug or disease. Notebook 01 is where that is measured; here it')
print('only sets how many pairs can be traced.')

,Gold pairs,Drugs resolved,Diseases resolved,Pairs mapped (in LCC),Mapped (%)
KG,,,,,
primekg,7254,9222,20817,5973,82.3
hetionet,7254,3039,136,813,11.2
drkg,7254,14166,3541,5633,77.7
openbilink,7254,67570,9254,4003,55.2
biokg,7254,9871,6870,5025,69.3
matrix,7254,998462,26998,6189,85.3


Unmapped pairs are a coverage fact, not a topology one: the graph does not
contain that drug or disease. Notebook 01 is where that is measured; here it
only sets how many pairs can be traced.


In [10]:
# Recovery by hop distance, on the full graph.
recovery_results = {}
for name in KG_NAMES:
    pairs = mapped_pairs[name]
    if not pairs:
        print(f'{name}: no mapped pairs, skipped')
        continue
    G = graphs[name]
    hops = []
    for src, tgt in pairs:
        try:
            d = nx.shortest_path_length(G, src, tgt)
            hops.append(d if d <= MAX_HOPS else None)
        except nx.NetworkXNoPath:
            hops.append(None)
    recovery = {h: sum(1 for d in hops if d is not None and d <= h) / len(pairs)
                for h in range(1, MAX_HOPS + 1)}
    recovery_results[name] = {'recovery': recovery, 'total_pairs': len(pairs),
                              'unreachable': sum(1 for d in hops if d is None)}
    print(f'{name:12s} ' + '  '.join(f'{h}-hop {recovery[h]:.1%}'
                                     for h in range(1, MAX_HOPS + 1))
          + f"  |  unreachable {recovery_results[name]['unreachable']:,}")

out.table(pd.DataFrame([{'KG': k, **{f'{h}-hop': v['recovery'][h] for h in v['recovery']},
                         'pairs': v['total_pairs'], 'unreachable': v['unreachable']}
                        for k, v in recovery_results.items()]), 'hop_recovery');

primekg      1-hop 31.4%  2-hop 80.4%  3-hop 97.9%  4-hop 99.8%  5-hop 100.0%  |  unreachable 0
hetionet     1-hop 58.9%  2-hop 89.8%  3-hop 99.9%  4-hop 100.0%  5-hop 100.0%  |  unreachable 0


drkg         1-hop 27.9%  2-hop 67.2%  3-hop 97.1%  4-hop 99.8%  5-hop 100.0%  |  unreachable 0


openbilink   1-hop 26.6%  2-hop 53.3%  3-hop 87.8%  4-hop 97.9%  5-hop 99.8%  |  unreachable 9


biokg        1-hop 33.2%  2-hop 84.7%  3-hop 98.2%  4-hop 99.8%  5-hop 100.0%  |  unreachable 1


matrix       1-hop 77.4%  2-hop 99.1%  3-hop 100.0%  4-hop 100.0%  5-hop 100.0%  |  unreachable 0


### Differential resilience gold standard vs random

In [11]:
# Differential resilience parameters.
#
# 2-hop reachability: it is what a drug-repurposing model actually traverses
# (drug -> intermediate -> disease), it stays tractable under repeated dropout,
# and the hop-recovery table above shows most gold pairs are within 2 hops.
N_SAMPLE    = 300          # gold and random pairs, balanced
DROP_RATES  = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
DR_MAX_HOPS = 2
N_TRIALS    = 3

In [12]:
# Gold vs type-matched random pairs under progressive edge dropout.
rng = np.random.RandomState(SEED)
diversity_results = {}

for name in KG_NAMES:
    pairs = mapped_pairs[name]
    if not pairs:
        print(f'{name}: no mapped pairs, skipped')
        continue

    G = graphs[name]
    edges, node_to_idx = topo.indexed_edges(G)

    # `pairs` and `negative_pairs` both work in node-label space; `edges` is in
    # row-position space. BOTH samples must be converted, not just the random
    # one. They agreed until now only because build_graph inserts nodes in idx
    # order, which makes the two spaces accidentally identical -- an accident
    # that would break silently on a KG whose node table is not 0..n-1 in order.
    _sampled = [pairs[i] for i in rng.choice(len(pairs), size=min(N_SAMPLE, len(pairs)),
                                             replace=False)]
    gold = [(node_to_idx[s], node_to_idx[t]) for s, t in _sampled
            if s in node_to_idx and t in node_to_idx]
    rand = topo.negative_pairs(G, LCC_SETS[name], ENTITY_TYPE_MAP[name],
                               set(pairs), len(gold), rng)
    rand = [(node_to_idx[s], node_to_idx[t]) for s, t in rand
            if s in node_to_idx and t in node_to_idx]

    # Balanced, so the two curves are estimated from the same number of pairs.
    k = min(len(gold), len(rand))
    if k == 0:
        print(f'{name}: no comparable random pairs, skipped')
        continue
    gold, rand = gold[:k], rand[:k]

    n_nodes = G.number_of_nodes()
    gold_curve, gold_trials = topo.dropout_curve(edges, gold, DROP_RATES, rng,
                                                 DR_MAX_HOPS, N_TRIALS, n_nodes)
    rand_curve, rand_trials = topo.dropout_curve(edges, rand, DROP_RATES, rng,
                                                 DR_MAX_HOPS, N_TRIALS, n_nodes)
    ri_gold = topo.resilience_index(DROP_RATES, gold_curve)
    ri_rand = topo.resilience_index(DROP_RATES, rand_curve)

    diversity_results[name] = {
        'drop_rates': DROP_RATES, 'gold_curve': gold_curve, 'rand_curve': rand_curve,
        'gold_trial_curves': gold_trials, 'rand_trial_curves': rand_trials,
        'ri_gold': ri_gold, 'ri_rand': ri_rand,
        'dr_score': topo.differential_resilience(ri_gold, ri_rand),
        'n_pairs': k,
    }
    print(f'{name:12s} pairs={k:>4}  RI_gold={ri_gold:.4f}  RI_rand={ri_rand:.4f}  '
          f"DR={diversity_results[name]['dr_score']:.4f}")
    del edges, node_to_idx
    gc.collect()

primekg      pairs= 300  RI_gold=0.7332  RI_rand=0.0577  DR=0.7169


hetionet     pairs= 300  RI_gold=0.8297  RI_rand=0.1703  DR=0.7947


drkg         pairs= 300  RI_gold=0.5676  RI_rand=0.0426  DR=0.5484


openbilink   pairs= 300  RI_gold=0.4442  RI_rand=0.0041  DR=0.4419


biokg        pairs= 300  RI_gold=0.8331  RI_rand=0.0742  DR=0.8197


matrix       pairs= 300  RI_gold=0.9826  RI_rand=0.0066  DR=0.9825


### Recovery curves and differential resilience

In [13]:
out.figure(fig04_recovery(recovery_results), 'hop_recovery');

  saved hop_recovery.pdf and .png (600 dpi)


In [14]:
out.figure(fig04_resilience(diversity_results), 'differential_resilience');

  saved differential_resilience.pdf and .png (600 dpi)


## 4. Community purity

In [15]:
# Louvain communities, and how well they line up with entity types.
#
# Cached per graph: Louvain on a large KG is expensive, and a cache means a
# kernel restart only recomputes the graphs that had not finished. The cache is
# keyed on node and edge counts AND on the networkx version that produced the
# partition, so neither a config change that alters a graph nor a different
# networkx (environment.yml pins only >=3.2,<4, and the Louvain implementation
# has changed inside that range) can silently serve a stale partition.
RESOLUTION = 1.0
_CACHE = BASE / 'results' / 'cache' / 'nb04_louvain.pkl'
_CACHE.parent.mkdir(parents=True, exist_ok=True)
community_results = pickle.loads(_CACHE.read_bytes()) if _CACHE.exists() else {}

for name, G in graphs.items():
    cached = community_results.get(name)
    _fresh = (cached is not None
              and cached.get('n_nodes') == G.number_of_nodes()
              and cached.get('n_edges') == G.number_of_edges()
              and cached.get('networkx_version') == nx.__version__
              and cached.get('ami_basis') == 'canonical')
    if _fresh:
        print(f"{name:12s} [cached] {cached['n_communities']:>5} communities  "
              f"AMI={cached['ami']:.4f}")
        continue
    if cached is not None:
        _why = ('graph changed'
                if (cached.get('n_nodes') != G.number_of_nodes()
                    or cached.get('n_edges') != G.number_of_edges())
                else f"built with networkx {cached.get('networkx_version')}, "
                     f"running {nx.__version__}"
                if cached.get('networkx_version') != nx.__version__
                else f"graded against {cached.get('ami_basis')} labels, "
                     f"now grading against canonical")
        print(f'{name:12s} [cache stale: {_why}] recomputing')

    # Graded against the canonical entity types, not the raw strings: BioKG
    # splits diseases into Disease (MeSH) and GeneticDisorder (OMIM), which the
    # config and notebook 01 already treat as one type.
    community_results[name] = topo.communities(
        G, seed=SEED, resolution=RESOLUTION, type_map=ENTITY_TYPE_MAP[name])
    cr = community_results[name]
    _CACHE.write_bytes(pickle.dumps(community_results))
    print(f"{name:12s} {cr['n_communities']:>5} communities  "
          f"AMI={cr['ami']:.4f}  NMI={cr['nmi']:.4f}  ARI={cr['ari']:.4f}  "
          f"Q={cr['modularity']:.4f}  ({cr['n_type_classes']} type classes)")

print('\nAMI is the scored statistic. NMI rises with the number of communities even')
print('when the partition is uninformative, so it would reward fragmentation.')
print(f'Louvain: networkx {nx.__version__}, resolution {RESOLUTION}, seed {SEED}, '
      f'partitioned on the largest connected component.')
for _n, _r in community_results.items():
    print(f'  {_n:12s} {_r["nodes_clustered"]:>9,} of {_r["n_nodes"]:>9,} nodes '
          f'partitioned into {_r["n_communities"]:,} communities')

# Schema-coverage guard, as in notebooks 01 and 02: a node type the config does
# not map is not silently folded in, it is named. These nodes stay in the graph
# and form their own class in the AMI, which is the honest treatment -- they are
# real nodes with a real type, just not one this benchmark has a canonical name
# for.
for name, cr in community_results.items():
    if cr.get('unmapped_types'):
        print(f"{name:12s} node types not mapped by config/kgs.yaml: "
              f"{', '.join(cr['unmapped_types'])} "
              f"(kept as their own class in the AMI)")

primekg      [cached]   576 communities  AMI=0.5444
hetionet     [cached]  1882 communities  AMI=0.2842


drkg           157 communities  AMI=0.2222  NMI=0.2247  ARI=0.1338  Q=0.4959  (13 type classes)


openbilink     186 communities  AMI=0.3342  NMI=0.3352  ARI=0.1988  Q=0.3868  (8 type classes)
biokg        [cached]  1529 communities  AMI=0.1569


matrix       15615 communities  AMI=0.3450  NMI=0.3468  ARI=0.0497  Q=0.6423  (36 type classes)

AMI is the scored statistic. NMI rises with the number of communities even
when the partition is uninformative, so it would reward fragmentation.
Louvain: networkx 3.6.1, resolution 1.0, seed 42, partitioned on the largest connected component.
  hetionet        47,031 of    47,031 nodes partitioned into 1,882 communities
  biokg          105,524 of   105,524 nodes partitioned into 1,529 communities
  primekg        129,375 of   129,375 nodes partitioned into 576 communities
  drkg            96,420 of    97,238 nodes partitioned into 157 communities
  openbilink     185,879 of   185,929 nodes partitioned into 186 communities
  matrix       7,316,507 of 7,354,612 nodes partitioned into 15,615 communities
biokg        node types not mapped by config/kgs.yaml: Complex (kept as their own class in the AMI)
openbilink   node types not mapped by config/kgs.yaml: GO (kept as their own class in the 

In [16]:
# What the largest communities are made of, as a curator aid.
for name, G in graphs.items():
    n2c = community_results[name]['node_to_community']
    sizes = {}
    for v in G.nodes():
        sizes.setdefault(n2c.get(v, -1), Counter())[G.nodes[v].get('node_type', 'unknown')] += 1
    print(f'\n{name} — largest communities')
    for c, types in sorted(sizes.items(), key=lambda kv: -sum(kv[1].values()))[:8]:
        total = sum(types.values())
        label, n = types.most_common(1)[0]
        print(f'  {c:>4d}: {total:>7,} nodes   dominant {label} ({n/total:.0%})')


primekg — largest communities
   248:  37,301 nodes   dominant gene/protein (67%)
    34:  35,597 nodes   dominant biological_process (64%)
   414:  26,122 nodes   dominant effect/phenotype (55%)
     1:   5,745 nodes   dominant drug (72%)
   158:   5,424 nodes   dominant molecular_function (99%)
   388:   3,027 nodes   dominant anatomy (100%)
   294:   2,689 nodes   dominant anatomy (100%)
   262:   2,037 nodes   dominant anatomy (100%)

hetionet — largest communities
    46:   9,633 nodes   dominant Gene (91%)
   519:   8,133 nodes   dominant Biological Process (48%)
  1126:   7,417 nodes   dominant Side Effect (76%)
   702:   5,700 nodes   dominant Biological Process (55%)
   742:   4,699 nodes   dominant Gene (59%)
  1122:   3,791 nodes   dominant Biological Process (38%)
  1318:   2,334 nodes   dominant Biological Process (37%)
  1200:   2,329 nodes   dominant Gene (39%)

drkg — largest communities
   125:  31,676 nodes   dominant Gene (54%)
   142:  25,895 nodes   dominant Gene 


openbilink — largest communities
   172:  38,900 nodes   dominant Drug (85%)
    79:  31,236 nodes   dominant Drug (41%)
    52:  18,147 nodes   dominant Phenotype (50%)
   175:  15,500 nodes   dominant Anatomy (87%)
    86:  15,348 nodes   dominant GO (54%)
    85:  15,093 nodes   dominant Gene (75%)
    89:  12,928 nodes   dominant GO (56%)
    93:   6,293 nodes   dominant Drug (49%)

biokg — largest communities
  1438:  40,576 nodes   dominant Gene/Protein (67%)
   200:  32,812 nodes   dominant Gene/Protein (40%)
   116:   3,413 nodes   dominant Gene/Protein (39%)
   508:   2,934 nodes   dominant Gene/Protein (85%)
  1431:   2,511 nodes   dominant Drug (69%)
  1074:   2,414 nodes   dominant Gene/Protein (70%)
  1396:   2,223 nodes   dominant Gene/Protein (65%)
    73:   2,221 nodes   dominant Drug (73%)



matrix — largest communities
  5184: 567,894 nodes   dominant gene/protein (94%)
  9336: 472,075 nodes   dominant molecular_function (51%)
  1299: 464,819 nodes   dominant drug (36%)
  9255: 435,576 nodes   dominant drug (53%)
  2989: 294,616 nodes   dominant drug (98%)
   868: 288,980 nodes   dominant drug (86%)
  12724: 267,619 nodes   dominant gene/protein (51%)
  13070: 176,653 nodes   dominant drug (98%)


In [17]:
# Modularity is returned by `topo.communities` alongside the agreement scores,
# computed on the same graph Louvain actually saw. It was previously recomputed
# here against the full graph, which raises NotAPartition whenever Louvain ran
# on the LCC subgraph.
modularity = {n: community_results[n]['modularity'] for n in graphs}
for n, q in modularity.items():
    print(f'  {n:12s} Q = {q:.4f}' + ('   (Q > 0.3: meaningful community structure)' if q > 0.3 else ''))

  primekg      Q = 0.5231   (Q > 0.3: meaningful community structure)
  hetionet     Q = 0.3470   (Q > 0.3: meaningful community structure)
  drkg         Q = 0.4959   (Q > 0.3: meaningful community structure)
  openbilink   Q = 0.3868   (Q > 0.3: meaningful community structure)
  biokg        Q = 0.4985   (Q > 0.3: meaningful community structure)
  matrix       Q = 0.6423   (Q > 0.3: meaningful community structure)


In [18]:
cp_df = pd.DataFrame([{
    'KG': name,
    'Communities': community_results[name]['n_communities'],
    'AMI (scored)': round(community_results[name]['ami'], 4),
    'NMI': round(community_results[name]['nmi'], 4),
    'ARI': round(community_results[name]['ari'], 4),
    'Q (modularity)': round(community_results[name]['modularity'], 4),
} for name in graphs]).set_index('KG')
out.table(cp_df.reset_index(), 'community_purity')
display(cp_df)

,Communities,AMI (scored),NMI,ARI,Q (modularity)
KG,,,,,
primekg,576,0.5444,0.5480,0.4486,0.5231
hetionet,1882,0.2842,0.3073,0.2132,0.3470
drkg,157,0.2222,0.2247,0.1338,0.4959
openbilink,186,0.3342,0.3352,0.1988,0.3868
biokg,1529,0.1569,0.1689,0.0696,0.4985
matrix,15615,0.3450,0.3468,0.0497,0.6423


## Dimension score

The topology dimension score combines four sub-metrics:

$$\text{Topology} = \frac{1}{4}\left(\text{Graph Cohesion} + \text{Clustering Coefficient} + \text{Graph Reachability} + \text{Community Purity}\right)$$

---

**Graph cohesion** is the LCC fraction directly:

$$\text{Graph Cohesion}_i = \frac{|\text{LCC}_i|}{N_i} \in [0, 1]$$

---

**Clustering coefficient** maps the empirical-to-random ratio via log scaling:

$$\text{Clustering Score}_i = \min\!\left(1,\; \frac{\log(C_i / C_{\text{ER},i})}{\log 1000}\right)$$

The ceiling at $C/C_{\text{ER}} = 1000$ represents 1000-fold enrichment over a random graph. A lower ceiling (e.g. 100) saturates at 1.0 for 5/6 KGs and provides no discriminative power. The 1000× ceiling gives meaningful spread across all five benchmark KGs (0.58-0.93). The log transform is appropriate because small-world enrichment scales multiplicatively. The raw $C$, $C_{\text{ER}}$, and $C/C_{\text{ER}}$ ratio are the primary reported statistics.

Calibration points:

| $C/C_{\text{ER}}$ | Score | Interpretation |
|----|----|----|
| 1 | 0.00 | No better than random |
| 10 | 0.33 | Moderate small-world |
| 100 | 0.67 | Strong small-world |
| 1000 | 1.00 | Saturates |

---

**Graph reachability** uses the **Differential Resilience Score**, a density-invariant measure of how much better the KG preserves externally validated drug-disease pairs under random edge dropout compared with random pairs:

$$\text{Recovery Score}_i = \frac{\text{RI}_{\text{gold},i} - \text{RI}_{\text{rand},i}}{\max\!(\varepsilon,\; 1 - \text{RI}_{\text{rand},i})}$$

where $\text{RI}$ is the **Resilience Index** (normalised area under the 2-hop recovery curve across dropout rates $r \in \{0, 0.1, 0.2, 0.3, 0.4, 0.5\}$):

$$\text{RI} = \frac{\int_0^{0.5} \text{Recovery}(r)\, dr}{0.5}$$

The numerator $(\text{RI}_{\text{gold}} - \text{RI}_{\text{rand}})$ captures the raw gap in structural resilience between validated and random pairs. The denominator $(1 - \text{RI}_{\text{rand}})$ normalises for graph density: a denser KG that keeps random pairs well-connected is penalised, ensuring the score rewards *specificity* of gold-standard connectivity rather than bulk edge redundancy.

**2-hop justification.** DR_MAX_HOPS=2 is used because (a) it is computationally tractable under repeated dropout, (b) 2-hop paths represent the minimal indirect connectivity (drug-intermediate-disease) relevant to drug repurposing, and (c) the 2-hop recovery actually achieved is reported per graph in the hop-recovery table rather than asserted here.

**Note on recovery curves and circularity.** The recovery curve at 1-hop is partially confounded by graph density: KGs that explicitly encode drug-disease edges (e.g. Hetionet with `treats`/`palliates`) will show higher 1-hop recovery by construction. This is not a structural property per se but a coverage property already measured in notebook 01. The DR score partially corrects for this by normalising against random-pair recovery at the same density.

---

**Community purity** is the Adjusted Mutual Information (AMI) between Louvain community assignments and entity type labels:

$$\text{Community Purity}_i = \text{AMI}(\text{Louvain communities}, \text{entity types}) \in [0, 1]$$

AMI corrects for chance (Vinh et al. 2010), unlike NMI which has known upward bias when the number of communities is large. NMI is still reported for interpretability but is not used for scoring. 5, 1.0, and 2.0 verifies that KG rankings are robust to this parameter choice.

Modularity Q is reported as a supplementary structural quality metric: Q > 0.3 indicates meaningful community structure.

> Vinh, N.X., Epps, J. & Bailey, J. "Information Theoretic Measures for Clusterings Comparison: Variants, Properties, Normalization and Correction for Chance." *JMLR* 11, 2837-2854 (2010).

In [19]:
# Dimension score: equal-weighted mean of the four sub-metrics.
#
# Every sub-metric used to fall back to 0 when its input was missing, so a graph
# whose clustering or community detection had failed scored zero and ranked last,
# indistinguishable from one that genuinely is poorly connected. A missing input
# is a broken run, not a finding.
_missing = {k: v for k, v in {
    'cohesion':         [n for n in KG_NAMES if n not in conn_df.index],
    'clustering':       [n for n in KG_NAMES if n not in clust_df.index],
    'reachability':     [n for n in KG_NAMES if n not in diversity_results],
    'community_purity': [n for n in KG_NAMES if n not in community_results],
}.items() if v}
if _missing:
    raise RuntimeError(
        'Cannot score: missing inputs for '
        + '; '.join(f'{k} -> {v}' for k, v in _missing.items())
        + '. Re-run the cells above rather than scoring these graphs 0, which would '
          'read as a property of the graph instead of a failed computation.')

sub_scores, raw_stats = {}, {}
for name in KG_NAMES:
    cl, dv, cm = clust_df.loc[name], diversity_results[name], community_results[name]
    sub_scores[name] = {
        'connectedness':    round(float(conn_df.loc[name, 'LCC fraction']), 4),
        # `small_world_score` and `differential_resilience` live in stats.topology
        # and are used both here and where the values are produced. They were
        # previously defined twice, and the resilience pair disagreed: the curve
        # cell stored an unclipped value while scoring clipped at 0.
        'small_world':      cl['SW score'],
        'reachability':     dv['dr_score'],
        'community_purity': round(cm['ami'], 4),
    }
    raw_stats[name] = {
        'lcc_fraction':  float(conn_df.loc[name, 'LCC fraction']),
        'C_empirical':   cl['C (empirical)'], 'C_se': cl['C SE'],
        'C_ER_random':   cl['C_ER (random)'], 'C_over_CER': cl['C / C_ER'],
        'sw_score_se':   cl['SW score SE'], 'nodes_sampled': int(cl['Nodes sampled']),
        'ri_gold':       dv['ri_gold'], 'ri_rand': dv['ri_rand'],
        'dr_pairs':      dv['n_pairs'],
        'community_ami': round(cm['ami'], 4), 'community_nmi': round(cm['nmi'], 4),
        'community_ari': round(cm['ari'], 4), 'modularity': round(cm['modularity'], 4),
        'n_communities': cm['n_communities'],
        'gold_pairs_mapped': len(mapped_pairs[name]),
        **{f'recovery_{h}hop': round(recovery_results[name]['recovery'][h], 4)
           for h in range(1, MAX_HOPS + 1) if name in recovery_results},
    }

summary_scores = {n: round(float(np.mean(list(sub_scores[n].values()))), 4)
                  for n in KG_NAMES}

scores = pd.DataFrame([{'KG': n, 'topology': summary_scores[n], **sub_scores[n]}
                       for n in KG_NAMES]).set_index('KG')
out.table(scores.reset_index(), 'dimension_scores')
out.table(pd.DataFrame([{'KG': n, **raw_stats[n]} for n in KG_NAMES]), 'raw_statistics')
display(scores)

out.checkpoint({
    'summary_scores': summary_scores,
    'sub_scores':     sub_scores,
    'raw_stats':      raw_stats,
    'conn_records':   conn_df.reset_index().to_dict('records'),
    'clust_records':  clust_df.reset_index().to_dict('records'),
    'recovery_results': recovery_results,
    # Curves kept; the large per-node structures are not.
    'diversity_scalars': {n: {k: v for k, v in r.items()
                              if k not in ('gold_trial_curves', 'rand_trial_curves')}
                          for n, r in diversity_results.items()},
    'community': {n: {k: v for k, v in community_results[n].items()
                      if k not in ('communities', 'node_to_community')}
                  for n in KG_NAMES},
})
print(out.summary())

,topology,connectedness,small_world,reachability,community_purity
KG,,,,,
primekg,0.7474,1.0000,0.7285,0.7169,0.5444
hetionet,0.6563,0.9602,0.5860,0.7947,0.2842
drkg,0.6405,0.9916,0.7999,0.5484,0.2222
openbilink,0.6755,0.9997,0.9263,0.4419,0.3342
biokg,0.6924,0.9330,0.8601,0.8197,0.1569
matrix,0.8306,0.9948,1.0000,0.9825,0.3450


04_topology: wrote 16 files
  results/checkpoints/04_topology.pkl
  results/figures/04_topology/clustering_coefficient.pdf
  results/figures/04_topology/clustering_coefficient.png
  results/figures/04_topology/differential_resilience.pdf
  results/figures/04_topology/differential_resilience.png
  results/figures/04_topology/graph_cohesion.pdf
  results/figures/04_topology/graph_cohesion.png
  results/figures/04_topology/hop_recovery.pdf
  results/figures/04_topology/hop_recovery.png
  results/tables/04_topology/clustering_coefficient.csv
  results/tables/04_topology/community_purity.csv
  results/tables/04_topology/dimension_scores.csv
  results/tables/04_topology/gold_pair_mapping.csv
  results/tables/04_topology/graph_cohesion.csv
  results/tables/04_topology/hop_recovery.csv
  results/tables/04_topology/raw_statistics.csv


In [20]:
# Figure-data cache: the inputs results/figures can be rebuilt from without
# reloading the raw KGs.
with open(BASE / 'results' / 'checkpoints' / '04_figdata.pkl', 'wb') as _f:
    pickle.dump({'conn_records': conn_df.reset_index().to_dict('records'),
               'clust_records': clust_df.reset_index().to_dict('records'),
               'recovery_results': recovery_results,
               'diversity_results': {n: {k: v for k, v in r.items()
                                         if 'trial_curves' not in k}
                                     for n, r in diversity_results.items()}}, _f)
print('figure-data cache -> 04_figdata.pkl')

figure-data cache -> 04_figdata.pkl


In [21]:
# Free KG state before the next notebook, which loads the same graphs again.
# Without this both sets stay resident and the kernel is OOM-killed.
for _v in ('kgs', 'graphs', 'community_results'):
    globals().pop(_v, None)
print(f'Released KG state; {gc.collect():,} objects collected.')
print('\nTopology: ' + '  '.join(f'{k} {v:.4f}' for k, v in summary_scores.items()))

Released KG state; 77,702,631 objects collected.

Topology: primekg 0.7474  hetionet 0.6563  drkg 0.6405  openbilink 0.6755  biokg 0.6924  matrix 0.8306
